# DC bias API tutorial

This tutorial uses the latest Qubex API to operate external DC voltage sources for JPA bias lines. All operations are available through `experiment.external_devices`. The primary path uses an NF ONS61797; a Qblox backend configuration is provided as an alternative.

> **This notebook writes voltages to live hardware.** Verify the wiring and allowed voltage range before connecting. Start only with an approved range near 0 V. The NF ONS61797 driver accepts 0–4 V and requires independent output mode (`OMD 0`).

## 1. Choose an `external_devices.yaml`

Use one of the following complete configurations as `<CONFIG_DIR>/external_devices.yaml`. Channel numbers in `wiring` are one-based.

### Primary: NF ONS61797

The primary tutorial path maps MUX 8 to ONS channel 9 over USB serial. For network transport, replace `port` with `ip_address`.

```yaml
devices:
  ONS1:
    driver: ons61797
    channels: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
    params:
      port: /dev/ttyACM0
      # For network transport, use this instead of port:
      # ip_address: 192.0.2.10

wiring:
  - mux: 0
    bias: ONS1-1

settings:
  ramp:
    rate_v_per_s: 0.1
    step_size_v: 0.01
    wait_s: 0.1
  readback:
    tolerance_v: 0.001
    max_attempts: 3
  reset_voltage: 0.0
```

### Alternative: Qblox SPI Rack D5a

Qubex connects as a TCP client through `qblox_backend`. Replace the host placeholder and ensure that no other client is using the backend.

```yaml
devices:
  Qblox1:
    driver: qblox_backend
    channels: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
    params:
      host: <qblox-backend-host>
      port: 52224
      timeout_s: 1200

wiring:
  - mux: 0
    bias: Qblox1-1

settings:
  ramp:
    rate_v_per_s: 0.1
    step_size_v: 0.01
    wait_s: 0.1
  readback:
    tolerance_v: 0.001
    max_attempts: 3
  reset_voltage: 0.0
```

`reset_voltage` is the reference value for initialization and shutdown. Set the resting point between measurements with the per-mux `idle_voltage` in `jpa_params.yaml`; when omitted, it falls back to `reset_voltage`.

## 2. `jpa_params.yaml`

```yaml
data:
  0:
    idle_voltage: 0.2
    bias_voltage: 1.04
    pump_frequency: 10.5
    pump_amplitude: 0.52
```

`bias_voltage` is the canonical name. The legacy `dc_voltage` key remains readable with a warning until at least v1.6.0, but should not be used in new configurations. Before applying a bias, Qubex validates both the bias and idle targets; no hardware write starts if either value is outside the driver's allowed range.

In [8]:
import time

import numpy as np
import qubex as qx

SYSTEM_ID = "64Qv3"
MUX_INDEX = 0  # NF primary config: ONS1 channel 9.
CONFIG_DIR = "qubex-config/64Qv3/config"
PARAMS_DIR = "qubex-config/64Qv3/params"

experiment = qx.Experiment(
    system_id=SYSTEM_ID,
    muxes=[MUX_INDEX],
    config_dir=CONFIG_DIR,
    params_dir=PARAMS_DIR,
)
experiment.connect()
ext = experiment.external_devices

date: 2026-08-20 12:03:45
python: 3.10.12
qubex: 1.5.0rc2+g91d3831c
env: /home/miyanaga/qubex/.venv
config: /home/miyanaga/qubex/tmp/qubex-config/64Qv3/config
params: /home/miyanaga/qubex/tmp/qubex-config/64Qv3/params
chip: 64Qv3 (2023-1st-64Q-No14-run3 chip (1,0))
qubits: ['Q00', 'Q01', 'Q02', 'Q03']
muxes: ['MUX00']
boxes: ['R23A']
Successfully connected.


## 3. Reset to a known state

Start with `reset_dc_voltages()`. It moves each target to `reset_voltage` and enables the output on devices with a physical switch. Every bulk write displays a confirmation prompt by default. Use `confirm=False` only in automation that performs equivalent safety checks separately.

In [9]:
states = ext.reset_dc_voltages(muxes=[MUX_INDEX])
for mux, state in states.items():
    print(mux, state)

You are going to reset the DC outputs to their reset voltages:

mux 0: +0.000 V

Do you want to continue?
 [y/n]:

0 DCVoltageState(mux_label='MUX00', mux_index=0, channel=1, voltage=0.0, output='on')
1 DCVoltageState(mux_label='MUX01', mux_index=1, channel=2, voltage=0.0, output='on')
2 DCVoltageState(mux_label='MUX02', mux_index=2, channel=3, voltage=0.0, output='on')
3 DCVoltageState(mux_label='MUX03', mux_index=3, channel=4, voltage=0.0, output='on')
4 DCVoltageState(mux_label='MUX04', mux_index=4, channel=5, voltage=0.0, output='on')
5 DCVoltageState(mux_label='MUX05', mux_index=5, channel=6, voltage=0.0, output='on')
6 DCVoltageState(mux_label='MUX06', mux_index=6, channel=7, voltage=0.0, output='on')
7 DCVoltageState(mux_label='MUX07', mux_index=7, channel=8, voltage=0.0, output='on')
8 DCVoltageState(mux_label='MUX08', mux_index=8, channel=9, voltage=0.0, output='on')
9 DCVoltageState(mux_label='MUX09', mux_index=9, channel=10, voltage=0.0, output='on')
10 DCVoltageState(mux_label='MUX10', mux_index=10, channel=11, voltage=0.0, output='on')
11 DCVoltageState(mux_label='MUX11', mux_index=11,

## 4. Read back the state

Readback does not change the voltage. The NF ONS61797 reports its channel voltage and physical output state. With the alternative Qblox configuration, the voltage is the setpoint retained by the backend rather than an independent measurement, and `output` is always `on`.

In [10]:
state = ext.get_dc_voltage_state(mux=MUX_INDEX)
print(state)

all_states = ext.get_dc_voltage_states()
print(all_states)

DCVoltageState(mux_label='MUX00', mux_index=0, channel=1, voltage=0.0, output='on')
{0: DCVoltageState(mux_label='MUX00', mux_index=0, channel=1, voltage=0.0, output='on'), 1: DCVoltageState(mux_label='MUX01', mux_index=1, channel=2, voltage=0.0, output='on'), 2: DCVoltageState(mux_label='MUX02', mux_index=2, channel=3, voltage=0.0, output='on'), 3: DCVoltageState(mux_label='MUX03', mux_index=3, channel=4, voltage=0.0, output='on'), 4: DCVoltageState(mux_label='MUX04', mux_index=4, channel=5, voltage=0.0, output='on'), 5: DCVoltageState(mux_label='MUX05', mux_index=5, channel=6, voltage=0.0, output='on'), 6: DCVoltageState(mux_label='MUX06', mux_index=6, channel=7, voltage=0.0, output='on'), 7: DCVoltageState(mux_label='MUX07', mux_index=7, channel=8, voltage=0.0, output='on'), 8: DCVoltageState(mux_label='MUX08', mux_index=8, channel=9, voltage=0.0, output='on'), 9: DCVoltageState(mux_label='MUX09', mux_index=9, channel=10, voltage=0.0, output='on'), 10: DCVoltageState(mux_label='MUX1

## 5. Apply a voltage to one mux

The context API is `external_devices.dc_voltage(...)`. `apply_voltage()` ramps from the current value to the target at the configured rate and verifies the readback. On both normal and exceptional context exit, the output ramps to `idle_voltage`.

In [11]:
TARGET_VOLTAGE_V = 1.0  # Change only to an approved value.

with ext.dc_voltage(mux=MUX_INDEX) as dc:
    applied = dc.apply_voltage(TARGET_VOLTAGE_V)
    print("applied:", applied)
    print("inside context:", dc.state)

print("after context:", ext.get_dc_voltage_state(mux=MUX_INDEX))

DC channel 1: ramping +0.000 V -> +1.000 V at 0.100 V/s.


applied: DCVoltageState(mux_label='MUX00', mux_index=0, channel=1, voltage=1.0, output='on')
inside context: DCVoltageState(mux_label='MUX00', mux_index=0, channel=1, voltage=1.0, output='on')


DC channel 1: ramping +1.000 V -> +0.200 V at 0.100 V/s.


after context: DCVoltageState(mux_label='MUX00', mux_index=0, channel=1, voltage=0.2, output='on')


## 6. Sweep

`sweep(sweep_range=...)` ramps through the values in order and yields the readback at each point. It does not return to idle between points; it returns to idle once when the context exits.

In [12]:
SWEEP_START_V = 0.0
SWEEP_STOP_V = 1.0
SWEEP_POINTS = 5
sweep_voltages = np.linspace(SWEEP_START_V, SWEEP_STOP_V, SWEEP_POINTS)

with ext.dc_voltage(mux=MUX_INDEX) as dc:
    started = time.perf_counter()
    for target_v, state in zip(
        sweep_voltages,
        dc.sweep(sweep_range=sweep_voltages),
        strict=True,
    ):
        print(f"{target_v:+.3f} V -> {state.voltage:+.6f} V")
    print(f"elapsed: {time.perf_counter() - started:.2f} s")

DC channel 1: ramping +0.200 V -> +0.000 V at 0.100 V/s.


+0.000 V -> +0.000000 V


DC channel 1: ramping +0.000 V -> +0.250 V at 0.100 V/s.


+0.250 V -> +0.250000 V


DC channel 1: ramping +0.250 V -> +0.500 V at 0.100 V/s.


...

DC channel 1: ramping +0.500 V -> +0.750 V at 0.100 V/s.


+0.750 V -> +0.750000 V


DC channel 1: ramping +0.750 V -> +1.000 V at 0.100 V/s.


+1.000 V -> +1.000000 V
elapsed: 31.59 s


DC channel 1: ramping +1.000 V -> +0.200 V at 0.100 V/s.


## 7. Apply calibrated biases during a measurement

`measurement.apply_dc_voltages()` resolves the muxes for the target qubits and applies their `bias_voltage` values from `jpa_params.yaml` for the duration of the measurement context. Uncalibrated muxes are skipped, and calibrated muxes return to idle on exit.

In [13]:
with experiment.measurement.apply_dc_voltages(experiment.qubit_labels):
    print("calibrated DC biases are active")
    # Run the JPA-assisted measurement here.

DC channel 1: ramping +0.200 V -> +1.040 V at 0.100 V/s.


calibrated DC biases are active


DC channel 1: ramping +1.040 V -> +0.200 V at 0.100 V/s.


## 8. Bulk operations

| API | Behavior |
| --- | --- |
| `reset_dc_voltages()` | Move to reset voltage and enable supported outputs |
| `bias_dc_voltages()` | Ramp to calibrated bias voltages |
| `idle_dc_voltages()` | Ramp to per-mux idle voltages |
| `shutdown_dc_voltages()` | Return to reset voltage and disable supported outputs |

On the primary NF ONS61797, shutdown ramps to the reset voltage and switches the output off. Because the alternative Qblox D5a has no physical output switch, shutdown only ramps it to the reset voltage.

In [14]:
bias_states = ext.bias_dc_voltages(muxes=[MUX_INDEX])
print("bias:", bias_states)

idle_states = ext.idle_dc_voltages(muxes=[MUX_INDEX])
print("idle:", idle_states)

You are going to ramp to the bias DC voltages:

mux 0: +1.040 V

Do you want to continue?
 [y/n]:

DC channel 1: ramping +0.200 V -> +1.040 V at 0.100 V/s.


bias: {0: DCVoltageState(mux_label='MUX00', mux_index=0, channel=1, voltage=1.04, output='on'), 1: DCVoltageState(mux_label='MUX01', mux_index=1, channel=2, voltage=0.0, output='on'), 2: DCVoltageState(mux_label='MUX02', mux_index=2, channel=3, voltage=0.0, output='on'), 3: DCVoltageState(mux_label='MUX03', mux_index=3, channel=4, voltage=0.0, output='on'), 4: DCVoltageState(mux_label='MUX04', mux_index=4, channel=5, voltage=0.0, output='on'), 5: DCVoltageState(mux_label='MUX05', mux_index=5, channel=6, voltage=0.0, output='on'), 6: DCVoltageState(mux_label='MUX06', mux_index=6, channel=7, voltage=0.0, output='on'), 7: DCVoltageState(mux_label='MUX07', mux_index=7, channel=8, voltage=0.0, output='on'), 8: DCVoltageState(mux_label='MUX08', mux_index=8, channel=9, voltage=0.0, output='on'), 9: DCVoltageState(mux_label='MUX09', mux_index=9, channel=10, voltage=0.0, output='on'), 10: DCVoltageState(mux_label='MUX10', mux_index=10, channel=11, voltage=0.0, output='on'), 11: DCVoltageState(m

You are going to ramp to the idle DC voltages:

mux 0: +0.200 V

Do you want to continue?
 [y/n]:

DC channel 1: ramping +1.040 V -> +0.200 V at 0.100 V/s.


idle: {0: DCVoltageState(mux_label='MUX00', mux_index=0, channel=1, voltage=0.2, output='on'), 1: DCVoltageState(mux_label='MUX01', mux_index=1, channel=2, voltage=0.0, output='on'), 2: DCVoltageState(mux_label='MUX02', mux_index=2, channel=3, voltage=0.0, output='on'), 3: DCVoltageState(mux_label='MUX03', mux_index=3, channel=4, voltage=0.0, output='on'), 4: DCVoltageState(mux_label='MUX04', mux_index=4, channel=5, voltage=0.0, output='on'), 5: DCVoltageState(mux_label='MUX05', mux_index=5, channel=6, voltage=0.0, output='on'), 6: DCVoltageState(mux_label='MUX06', mux_index=6, channel=7, voltage=0.0, output='on'), 7: DCVoltageState(mux_label='MUX07', mux_index=7, channel=8, voltage=0.0, output='on'), 8: DCVoltageState(mux_label='MUX08', mux_index=8, channel=9, voltage=0.0, output='on'), 9: DCVoltageState(mux_label='MUX09', mux_index=9, channel=10, voltage=0.0, output='on'), 10: DCVoltageState(mux_label='MUX10', mux_index=10, channel=11, voltage=0.0, output='on'), 11: DCVoltageState(mu

## 9. Finish the session

For a normal exit, return to idle before disconnecting. For maintenance or chip replacement, call `shutdown_dc_voltages()` when appropriate.

In [15]:
ext.idle_dc_voltages(muxes=[MUX_INDEX])
print("final:", ext.get_dc_voltage_state(mux=MUX_INDEX))
experiment.disconnect()

You are going to ramp to the idle DC voltages:

mux 0: +0.200 V

Do you want to continue?
 [y/n]:

final: DCVoltageState(mux_label='MUX00', mux_index=0, channel=1, voltage=0.2, output='on')


# 10. Shutdown all DC voltages.

In [17]:
ext.shutdown_dc_voltages()

You are going to ramp to the reset voltages and turn off the DC outputs:

mux 0: +0.000 V
mux 1: +0.000 V
mux 2: +0.000 V
mux 3: +0.000 V
mux 4: +0.000 V
mux 5: +0.000 V
mux 6: +0.000 V
mux 7: +0.000 V
mux 8: +0.000 V
mux 9: +0.000 V
mux 10: +0.000 V
mux 11: +0.000 V
mux 12: +0.000 V
mux 13: +0.000 V
mux 14: +0.000 V
mux 15: +0.000 V

Do you want to continue?
 [y/n]:

DC channel 1: ramping +0.200 V -> +0.000 V at 0.100 V/s.
DC channel 1: output switched off.
DC channel 2: output switched off.
DC channel 3: output switched off.
DC channel 4: output switched off.
DC channel 5: output switched off.
DC channel 6: output switched off.
DC channel 7: output switched off.
DC channel 8: output switched off.
DC channel 9: output switched off.
DC channel 10: output switched off.
DC channel 11: output switched off.
DC channel 12: output switched off.
DC channel 13: output switched off.
DC channel 14: output switched off.
DC channel 15: output switched off.
DC channel 16: output switched off.


{0: DCVoltageState(mux_label='MUX00', mux_index=0, channel=1, voltage=0.0, output='off'),
 1: DCVoltageState(mux_label='MUX01', mux_index=1, channel=2, voltage=0.0, output='off'),
 2: DCVoltageState(mux_label='MUX02', mux_index=2, channel=3, voltage=0.0, output='off'),
 3: DCVoltageState(mux_label='MUX03', mux_index=3, channel=4, voltage=0.0, output='off'),
 4: DCVoltageState(mux_label='MUX04', mux_index=4, channel=5, voltage=0.0, output='off'),
 5: DCVoltageState(mux_label='MUX05', mux_index=5, channel=6, voltage=0.0, output='off'),
 6: DCVoltageState(mux_label='MUX06', mux_index=6, channel=7, voltage=0.0, output='off'),
 7: DCVoltageState(mux_label='MUX07', mux_index=7, channel=8, voltage=0.0, output='off'),
 8: DCVoltageState(mux_label='MUX08', mux_index=8, channel=9, voltage=0.0, output='off'),
 9: DCVoltageState(mux_label='MUX09', mux_index=9, channel=10, voltage=0.0, output='off'),
 10: DCVoltageState(mux_label='MUX10', mux_index=10, channel=11, voltage=0.0, output='off'),
 11: D

# 11. Reset all DC voltages.

In [18]:
ext.reset_dc_voltages()

You are going to reset the DC outputs to their reset voltages:

mux 0: +0.000 V
mux 1: +0.000 V
mux 2: +0.000 V
mux 3: +0.000 V
mux 4: +0.000 V
mux 5: +0.000 V
mux 6: +0.000 V
mux 7: +0.000 V
mux 8: +0.000 V
mux 9: +0.000 V
mux 10: +0.000 V
mux 11: +0.000 V
mux 12: +0.000 V
mux 13: +0.000 V
mux 14: +0.000 V
mux 15: +0.000 V

Do you want to continue?
 [y/n]:

DC channel 1: output switched on at +0.000 V.
DC channel 2: output switched on at +0.000 V.
DC channel 3: output switched on at +0.000 V.
DC channel 4: output switched on at +0.000 V.
DC channel 5: output switched on at +0.000 V.
DC channel 6: output switched on at +0.000 V.
DC channel 7: output switched on at +0.000 V.
DC channel 8: output switched on at +0.000 V.
DC channel 9: output switched on at +0.000 V.
DC channel 10: output switched on at +0.000 V.
DC channel 11: output switched on at +0.000 V.
DC channel 12: output switched on at +0.000 V.
DC channel 13: output switched on at +0.000 V.
DC channel 14: output switched on at +0.000 V.
DC channel 15: output switched on at +0.000 V.
DC channel 16: output switched on at +0.000 V.


{0: DCVoltageState(mux_label='MUX00', mux_index=0, channel=1, voltage=0.0, output='on'),
 1: DCVoltageState(mux_label='MUX01', mux_index=1, channel=2, voltage=0.0, output='on'),
 2: DCVoltageState(mux_label='MUX02', mux_index=2, channel=3, voltage=0.0, output='on'),
 3: DCVoltageState(mux_label='MUX03', mux_index=3, channel=4, voltage=0.0, output='on'),
 4: DCVoltageState(mux_label='MUX04', mux_index=4, channel=5, voltage=0.0, output='on'),
 5: DCVoltageState(mux_label='MUX05', mux_index=5, channel=6, voltage=0.0, output='on'),
 6: DCVoltageState(mux_label='MUX06', mux_index=6, channel=7, voltage=0.0, output='on'),
 7: DCVoltageState(mux_label='MUX07', mux_index=7, channel=8, voltage=0.0, output='on'),
 8: DCVoltageState(mux_label='MUX08', mux_index=8, channel=9, voltage=0.0, output='on'),
 9: DCVoltageState(mux_label='MUX09', mux_index=9, channel=10, voltage=0.0, output='on'),
 10: DCVoltageState(mux_label='MUX10', mux_index=10, channel=11, voltage=0.0, output='on'),
 11: DCVoltageSta